In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:13pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:80px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:2px;}
table.dataframe{font-size:5xpt;} 
</style>
"""))

# ※ Quiz : 경주여행과 전주여행에 대해 최빈단어 시각화와 유사도 분석


## 1) naver open API를 활용하여 블로그에 "경주여행", "전주여행"을 각가 500건씩 검색하여 백업(data/quiz/naver.csv)
   * 파일 내용(query, no, title, link, description, total_text(title + ' ' + descriprtion) 

### 1. 네이버 open API 활용하여 검색 추출
- query, no, title, link, description, total_text(title + ' ' + descriprtion)

In [8]:
# .env 가져오기
from dotenv import load_dotenv
import os
load_dotenv()

True

In [21]:
# 네이버 개발자 센터에 있는 소스
import os
import sys
import urllib.request
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("경주 여행")
url = "https://openapi.naver.com/v1/search/blog.json?query=" + encText # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8')[:200])
    #item = json.loads(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Thu, 03 Sep 2026 17:59:10 +0900",
	"total":2735137,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"3월의 <b>경주여행<\/b>.",
			"link":"https:\/\/lje77777.tistory.com\/7132",
			"


In [22]:
# 문자 → dict
import json
from html import unescape #descriptond에 있는 &lt;(특수문자)를 <로 변경
import requests
import pandas as pd
import re # 특수문자 제외 @@ _._

In [25]:
query = "경주 여행"
start = 1

# url = f"https://openapi.naver.com/v1/search/blog.json?query={query}&display=100&start={start}"
url = "https://openapi.naver.com/v1/search/blog.json"
params = {'query':query,
          'display':100,
          'start':start}
headers = {"X-Naver-Client-Id":client_id,
           "X-Naver-Client-Secret":client_secret}
response = requests.get(url, headers=headers, params=params)

# 문자 → dict
# items = json.loads(response.text)['items']
items = response.json()['items']
items

[{'title': '<b>경주여행</b> 1박 2일.',
  'link': 'https://mauntbaek.tistory.com/4293292',
  'description': '&lt;<b>경주여행</b>&gt;1박 2일 <b>경주</b>를 들어서자 마자 온 천지가 벚꽃으로 뒤덮혀 아름다운 풍경에 입을 다물 수가... 단체로 온 <b>여행</b>이라서 주체측의 시간표대로 움직여줘야 했다. 호텔로 와서 다시 보문호수가를 돌았다.... ',
  'bloggername': 'mauntbaek.tistory.com - 산의 향기를 찾아서 :: mauntbaek.tistory.com - 산의 향기를 찾아서',
  'bloggerlink': 'https://mauntbaek.tistory.com/',
  'postdate': '20150412'},
 {'title': '[<b>경주여행</b>]양동마을 무첨당',
  'link': 'https://alame.tistory.com/149',
  'description': '무첨당, 양동마을 무첨당, <b>경주</b> 양동마을 무첨당, <b>경주여행</b> 양동마을 무첨당, <b>경주</b> 가볼만한곳 양동마을 무첨당 <b>경주</b> 양동마을 무첨당은 조선시대 성리학자이자 문신이었던 회재 이언적 선생 종택의 일부로 조선... ',
  'bloggername': '알라와 함께 떠나는 여행',
  'bloggerlink': 'https://alame.tistory.com/',
  'postdate': '20140615'},
 {'title': '[경북/<b>경주여행</b>]천문 기상 관측대, 첨성대',
  'link': 'https://kwon-blog.tistory.com/1574',
  'description': '*<b>여행</b>일자: 2016년 9월 7일(목) *<b>여행</b>인원: 친구와 함께 첨성대 경북 <b>경주</b>시 인왕동 839-1 전화: 054-779-8741 첨성대 관람료는 무료! ☞ [경북/<b

In [27]:
# title과 description의 <b>없애기, html의 특수문자 없애기, 일반특수문자 없애기
item = items[4]
title = item['title'].replace('<b>', ' ').replace('</b>', ' ')
title = unescape(title)
title = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)
description = item['description'].replace('<b>', ' ').replace('</b>', ' ')
description = unescape(description)
description = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', description)
link = item['link']
totaltext = title + ' ' + description
print(totaltext)
print(link)

 경주여행  야경이 이쁜 안압지 첨성대 16년3월31일  어울린다  경주 여행 에서 빠질수 없는 관광지 중 하나인 첨성대   첨성대는 신라시대에 별을 관측하기        최근  경주 를  여행 하는 이들이 꼭 빼놓지 않는  여행 코스가 있다  다름 아닌  경주 의 야경을 둘러보는 것     
https://skdywjd25.tistory.com/4030


In [28]:
# re 정규표현식을 이용해서 특수문자 없애기
title = '[여행] ## & ktx 타고 짱 ㅋㅋ ㅠㅠ'
re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)

' 여행       ktx 타고 짱      '

In [29]:
# 네이버 API 정보 및 검색 정보
from dotenv import load_dotenv
import os
load_dotenv()
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

queries = ['경주 여행', '전주 여행']
max_start = 5


In [ ]:
def get_search_element_save(query, start):
    'query와 start로 naver 블로그 검색한 결과로 title, link, descript, totacl_text를 dict_list'
    pass


In [ ]:
for query in queries:
    for start in range(1, max_start+1):
        get_search_item_return(query, start)
        'header와 params와 url로 request.get(url) → items → title, link, description, total_text'
        '함수화'

## 2) naver.csv에서 total_text를 품사 태깅(naver_pos.csv)
   * 파일 내용 : query, no, token, pos

## 3) 명사만 추출(naver_pos_nouns.csv)
   * query, token, pos

## 4) 빈도분석 백업(naver_pos_nouns_count.csv)
   * token, 경주빈도, 전주빈도, 빈도합

## 5) 빈도 시각화(wordcloud, Text.plot)
   * 이미지 저장


## 6) 단어간 거리 분석(Word2Vec)